# Spark 日销量汇总

读取订单 JSON，按 SKU 汇总销量与 GMV，并覆盖 Iceberg 对应日期分区。

## 运行条件

运行需要 Spark、Java、Iceberg catalog，以及相应存储连接器和访问权限，配置见 [[processing/work/spark/README|Spark 运行方式]]。输入至少包含 `event_type`、`sku_id`、`quantity`、`pay_amount` 或 `amount`。将 `run_job` 改为 `True` 才写入；同日期分区会被覆盖。运行后查询目标表，核对 SKU 汇总与日期分区。当前示例仅按事件类型筛选，实际支付成功口径需结合业务状态确认。


In [ ]:
from __future__ import annotations
import argparse
from typing import Any


## 实现


In [ ]:
def amount_column(raw: Any) -> Any:
    from pyspark.sql.functions import col

    if "pay_amount" in raw.columns:
        return col("pay_amount")
    return col("amount")


In [ ]:
def build_sales_table(spark: Any, input_path: str, biz_date: str) -> Any:
    from pyspark.sql.functions import col, lit, sum as spark_sum

    raw = spark.read.json(input_path)
    paid_orders = raw.filter(col("event_type").isin("order_paid", "payment_result"))
    return (
        paid_orders.select(
            col("sku_id"),
            col("quantity").cast("long").alias("quantity"),
            amount_column(raw).cast("double").alias("amount"),
        )
        .filter(col("sku_id").isNotNull() & (col("quantity") > 0))
        .groupBy("sku_id")
        .agg(
            spark_sum("quantity").alias("sales_qty"),
            spark_sum("amount").alias("gmv"),
        )
        .withColumn("dt", lit(biz_date))
    )


In [ ]:
def namespace_of(table: str) -> str:
    parts = table.split(".")
    if len(parts) < 3:
        raise ValueError("表名需要包含 catalog.namespace.table，例如 lakehouse.dws.sku_sales_daily")
    return ".".join(parts[:-1])


In [ ]:
def ensure_table(spark: Any, table: str) -> None:
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {namespace_of(table)}")
    spark.sql(
        f"""
        CREATE TABLE IF NOT EXISTS {table} (
          sku_id STRING,
          sales_qty BIGINT,
          gmv DOUBLE,
          dt STRING
        )
        USING iceberg
        PARTITIONED BY (dt)
        """
    )


In [ ]:
def run(input_path: str, table: str, biz_date: str) -> None:
    from pyspark.sql import SparkSession

    spark = SparkSession.builder.appName("sku-sales-iceberg").getOrCreate()
    try:
        ensure_table(spark, table)
        sales = build_sales_table(spark, input_path, biz_date)
        sales.writeTo(table).overwritePartitions()
    finally:
        spark.stop()


## 本地检查

此单元不连接外部服务。


In [ ]:
namespace_of("lakehouse.dws.sku_sales_daily")


## 连接服务

先修改地址与参数，再开启执行开关。


In [ ]:
input_path = "s3a://raw/orders/dt=2026-07-10/*.json"
table = "lakehouse.dws.sku_sales_daily"
biz_date = "2026-07-10"
run_job = False

if run_job:
    run(input_path, table, biz_date)


## 命令行入口

导出为脚本后使用；在 Notebook 内运行仅定义入口，不解析内核参数。


In [ ]:
def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", required=True)
    parser.add_argument("--table", default="lakehouse.dws.sku_sales_daily")
    parser.add_argument("--date", required=True)
    args = parser.parse_args()
    run(args.input, args.table, args.date)


if __name__ == "__main__" and "get_ipython" not in globals():
    main()
